In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

!wget $data -O data-week-3.csv

df = pd.read_csv('data-week-3.csv')
df.columns = df.columns.str.lower().str.replace(' ','_')
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ','_')

df.totalcharges = pd.to_numeric(df.totalcharges,errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

--2026-10-21 20:34:14--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  1.49MB/s    in 0.6s    

2026-10-21 20:34:15 (1.49 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [3]:
df_full_train,df_test = train_test_split(df,test_size=0.2,random_state=1)

In [4]:
numerical = ['tenure','monthlycharges','totalcharges']
categorical = [ 'gender', 'seniorcitizen', 'partner', 'dependents',
        'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [5]:
def train(df_train, y_train,C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C , max_iter=5000)
    model.fit(X_train, y_train)
    
    return dv, model

In [6]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [7]:
C=1.0
n_splits=5

In [8]:
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)
print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=1.0 0.843 +- 0.012


In [9]:
scores

[0.8497527153602723,
 0.8423267326732673,
 0.8584289819444214,
 0.8333388198502181,
 0.8247208436724566,
 0.8410755910755912,
 0.8431129925105829,
 0.8255139152435167,
 0.8458425967096488,
 0.8611139121374071]

In [10]:
dv,model = train(df_full_train,df_full_train.churn.values,C=1.0)
y_pred = predict(df_test,dv,model)
y_test = df_test.churn.values
auc = roc_auc_score(y_test,y_pred)
auc

0.8584167506256296

### Save the model

In [11]:
import pickle

In [12]:
output_file = f'model_C={C}.bin'
output_file

'model_C=1.0.bin'

In [13]:
with open(output_file,'wb') as f_out:
    pickle.dump((dv,model),f_out)

### Load the model

In [1]:
import pickle

In [2]:
model_file = 'model_C=1.0.bin'

In [3]:
with open(model_file,'rb') as f_in:
    dv,model = pickle.load(f_in)

In [4]:
dv,model

(DictVectorizer(sparse=False), LogisticRegression(max_iter=5000))

In [7]:
customer = {
    'gender':'female',
    'seniorcitizen':0,
    'partner':'yes',
    'dependents':'no',
    'phoneservice':'no',
    'multiplelines':'no_phone_service',
    'internetservice':'dsl',
    'onlinesecurity':'no',
    'onlinebackup':'yes',
    'deviceprotection':'no',
    'techsupport':'no',
    'streamingtv':'no',
    'streamingmovies':'no',
    'contract':'month-to-month',
    'paperlessbilling':'yes',
    'paymentmethod':'electronic_check',
    'tenure':1,
    'monthlycharges':29.85,
    'totalcharges':29.85
}
    


In [10]:
X = dv.transform([customer])

In [11]:
model.predict_proba(X)[0,1]

np.float64(0.628213609423926)